# PitchLens — revue humaine des annotations

Ce notebook sert uniquement à prendre les décisions humaines nécessaires pour construire `v002`. Il ne modifie jamais `raw` ni `v001`. Chaque clic est enregistré dans `data/processed/v001/qa_review_decisions.csv`, ce qui permet de fermer le notebook et de reprendre plus tard.

## Ordre conseillé

La file couvre désormais **les 7 152 images sources d'Infactory**, et non seulement les 838 images signalées initialement. Les décisions déjà enregistrées sont conservées. SoccerEye reste un dataset séparé et n'est pas inclus dans cette interface.

1. terminer les **1 069 images test** ;
2. terminer les **1 080 images validation** ;
3. revoir ensuite les **5 003 images train** ;
4. générer `v002` seulement lorsque les décisions nécessaires sont terminées.

Le filtre par défaut affiche toutes les images, dans l'ordre test, validation, train, puis par séquence et numéro de frame. Une absence de signal automatique ne signifie pas que l'annotation est correcte.

## Actions de revue

| Bouton | Effet futur dans `v002` | Quand l'utiliser |
|---|---|---|
| **Valider et avancer** | conserve l'annotation affichée, y compris une correction déjà enregistrée | l'annotation est correcte |
| **Aucun ballon** | écrit un label vide | aucun ballon n'est visible |
| **Garder uniquement cette boîte** | supprime les autres boîtes rouges et conserve celle choisie | image avec deux ou plusieurs boîtes dont une seule est correcte |
| **Sauver la boîte dessinée** | supprime toutes les anciennes boîtes et les remplace par la boîte dessinée | ballon visible mais boîte absente, fausse ou imprécise |
| **Exclure l'image** | retire l'image de v002 | décision impossible : occultation, ambiguïté ou image inutilisable |

Les rectangles rouges sont les annotations d'origine. Pour un doublon, choisis la bonne boîte dans la liste puis clique sur **Garder uniquement cette boîte**. Pour redessiner : utilise d'abord le zoom si nécessaire, clique ensuite sur **✏️ Activer le dessin** ; les rouges disparaissent alors de l'image centrale. Trace le nouveau rectangle puis sauvegarde-le. Le bouton est indispensable car le zoom et le dessin utilisent le même geste de souris. Les images précédente et suivante donnent le contexte temporel mais ne sont pas modifiées.

In [ ]:
%matplotlib widget

from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pitchlens.data.review as review_module

# Recharge le module si cette cellule est rejouée après une amélioration de l'interface.
review_module = importlib.reload(review_module)
ReviewStore = review_module.ReviewStore
ReviewWidget = review_module.ReviewWidget

PROCESSED_ROOT = PROJECT_ROOT / "data/processed/v001"
store = ReviewStore(PROCESSED_ROOT)
store.progress()

## Revue par séquence

**Première / Dernière** rejoignent immédiatement les extrémités de la séquence. Le curseur **Image** permet d'aller directement à n'importe quelle image : glisse-le ou clique sur sa valeur pour saisir un numéro (position de 1 à N dans la séquence).

**Exclure toute la séquence** écarte toutes ses images de la future v002, y compris celles déjà corrigées. Les décisions précédentes sont sauvegardées dans `data/processed/v001/sequence_backups/`. **Annuler l'exclusion totale** les restaure ; les images qui n'avaient pas de décision redeviennent à revoir. Les décisions individuelles modifiées après l'exclusion restent conservées.

Choisis une séquence dans la liste : son nombre d'images restant à revoir est affiché. Si elle est trop dégradée, **Écarter les N restantes** exclut toutes ses images sans décision de la future v002, en conservant tes corrections et validations existantes. Cela inclut les images que tu avais sautées plus tôt dans cette séquence. L'interface passe ensuite à la prochaine image sans décision.

Pour revenir sur ce choix, sélectionne à nouveau la séquence et clique sur **Rétablir la suite écartée**. Seules les exclusions groupées non retouchées redeviennent à revoir. Les compteurs distinguent les décisions individuelles des exclusions par séquence.

La revue reprend à la première image sans décision. Les images restent dans un ordre stable : **Précédente** revient sur une image même déjà validée, **Suivante** avance sans enregistrer. **Prochaine à revoir** saute aux décisions restantes.

Trois actions principales : **Valider et avancer**, **Aucun ballon**, **Exclure l'image**. Chaque action sauvegarde et avance ; tu peux revenir et remplacer ta décision. **Corriger les boîtes** ouvre les outils de dessin et de sélection. Les notes et l'origine sont dans des panneaux repliables.

In [ ]:
# Ferme l'ancienne interface avant de la recréer après une mise à jour.
if "review" in globals():
    review.selector.disconnect_events()
    review.plt.close(review.figure)
    review.ui.close()
review = ReviewWidget(store)
review.display()

## Vérifier la progression

Cette cellule relit les compteurs. Le fichier de décisions est local et ignoré par Git, car il accompagne le dataset plutôt que le code. Il sera utilisé plus tard par la génération reproductible de `v002`.

In [ ]:
store.progress()